## 08 - Wrap‑up: Findings, Limitations, and Mitigations

This notebook **summarizes the key results** from Notebooks **01–07** and turns them into:
- a coherent set of **final conclusions** about *bias, under‑representation, and (short‑term) reinforcement*;
- a **requirements checklist** (what is already covered vs. what is missing);
- a short, justified set of **mitigation ideas** that can be described in my report.

### Setup and Data Loading

Files from previous notebooks and Imports necessary

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_processed_dir():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        p = base / "data" / "processed"
        if p.exists():
            return p
    raise FileNotFoundError("Could not find data/processed by walking up from the current working directory.")

PROCESSED = find_processed_dir()
print("Using PROCESSED =", PROCESSED)


def standardize_clicks(df: pd.DataFrame) -> pd.DataFrame:
    cols = set(df.columns)

    imp_col = "impression_id" if "impression_id" in cols else ("imp_id" if "imp_id" in cols else None)
    uid_col = "user_id" if "user_id" in cols else ("uid" if "uid" in cols else None)
    nid_col = "news_id" if "news_id" in cols else ("nid" if "nid" in cols else None)
    clk_col = "clicked" if "clicked" in cols else ("click" if "click" in cols else None)

    assert imp_col and uid_col and nid_col and clk_col, f"Missing expected columns in: {df.columns}"

    out = df[[imp_col, uid_col, nid_col, clk_col]].copy()
    out.columns = ["impression_id", "user_id", "news_id", "clicked"]
    return out


clicks_train = pd.read_pickle(PROCESSED / "clicks_train.pkl")
clicks_dev   = pd.read_pickle(PROCESSED / "clicks_dev.pkl")
news_train   = pd.read_pickle(PROCESSED / "news_train.pkl")
news_dev     = pd.read_pickle(PROCESSED / "news_dev.pkl")

clicks_train = standardize_clicks(clicks_train)
clicks_dev   = standardize_clicks(clicks_dev)

print("clicks_train:", clicks_train.shape, " | clicks_dev:", clicks_dev.shape)
print("news_train :", news_train.shape,  " | news_dev  :", news_dev.shape)

pol_train_path = PROCESSED / "political_ideology_train.pkl"
pol_dev_path   = PROCESSED / "political_ideology_dev.pkl"
pol_train = pd.read_pickle(pol_train_path) if pol_train_path.exists() else None
pol_dev   = pd.read_pickle(pol_dev_path)   if pol_dev_path.exists()   else None
print("Loaded pol_train:", pol_train is not None, "| pol_dev:", pol_dev is not None)


### Rebuild the DEV “impressions” table

Reconstruct, per **DEV impression**:
- the **candidate slate** (all shown candidates),
- the **clicked items** (ground truth for evaluation),
- the **user id**.

This lets us compute exposure statistics **on the full slate** and, when useful, on **Top‑K** after applying a scoring function.

In [ ]:
impression_candidates = clicks_dev.groupby("impression_id")["news_id"].apply(list)
imp_user = clicks_dev.groupby("impression_id")["user_id"].first()

clicked_by_imp = clicks_dev.loc[clicks_dev["clicked"] == 1].groupby("impression_id")["news_id"].apply(list)
clicked_by_imp = clicked_by_imp.reindex(impression_candidates.index, fill_value=[])

print("DEV impressions:", len(impression_candidates))
print("Mean #candidates:", np.mean([len(x) for x in impression_candidates.values]).round(2))
print("Share with ≥1 click:", np.mean([len(x) > 0 for x in clicked_by_imp.values]).round(4))


### Define “politics” and “engaged users” (from TRAIN)
As done in previous notebooks

In [ ]:
news_train["is_politics"] = news_train["subcategory"].astype(str).str.contains("polit", case=False, na=False)
news_dev["is_politics"]   = news_dev["subcategory"].astype(str).str.contains("polit", case=False, na=False)

polit_ids_train = set(news_train.loc[news_train["is_politics"], "news_id"])
polit_ids_dev   = set(news_dev.loc[news_dev["is_politics"],   "news_id"])

# Engaged users: >= 3 TRAIN political clicks
train_pol_clicks = clicks_train.loc[(clicks_train["clicked"] == 1) & (clicks_train["news_id"].isin(polit_ids_train))]
pol_clicks_per_user = train_pol_clicks.groupby("user_id").size()

ENGAGED_THRESHOLD = 3
engaged_users = set(pol_clicks_per_user[pol_clicks_per_user >= ENGAGED_THRESHOLD].index)

print("TRAIN users:", clicks_train["user_id"].nunique())
print("DEV users  :", clicks_dev["user_id"].nunique())
print("Engaged users (>=3 TRAIN political clicks):", len(engaged_users))

def user_group(uid):
    return "engaged" if uid in engaged_users else "other"


### Baseline scoring functions

We keep this simple and reproducible:

- **Popularity baseline:** score = number of TRAIN clicks for the news item.
- **Category‑preference baseline:** estimate each user's preference distribution over categories from TRAIN clicks, then score(item) = pref(user, item.category).

These are enough to demonstrate *exposure imbalance* and test a mitigation.

In [ ]:
pop_score = clicks_train.loc[clicks_train["clicked"] == 1].groupby("news_id").size().to_dict()

def pop_score_fn(imp_id, news_id):
    return float(pop_score.get(news_id, 0.0))

train_meta = news_train.set_index("news_id")[["category","subcategory"]]

train_clicks_only = clicks_train.loc[clicks_train["clicked"] == 1, ["user_id","news_id"]].copy()
train_clicks_only = train_clicks_only.join(train_meta, on="news_id", how="left")

user_cat_counts = (
    train_clicks_only.groupby(["user_id","category"])
    .size()
    .rename("cnt")
    .reset_index()
)

user_total = user_cat_counts.groupby("user_id")["cnt"].transform("sum")
user_cat_counts["pref"] = user_cat_counts["cnt"] / user_total

user_pref_cat = {
    uid: dict(zip(chunk["category"], chunk["pref"]))
    for uid, chunk in user_cat_counts.groupby("user_id")
}

dev_meta = news_dev.set_index("news_id")[["category","subcategory"]]

def cat_personal_score_fn(imp_id, news_id):
    uid = imp_user.get(imp_id, None)
    if uid is None or news_id not in dev_meta.index:
        return 0.0
    cat = dev_meta.loc[news_id, "category"]
    return float(user_pref_cat.get(uid, {}).get(cat, 0.0))

print("Popularity items with score:", len(pop_score))
print("Users with learned category preferences:", len(user_pref_cat))
